# BERT fine-tune（階段 8，開發規格 7.27）

在 Colab 執行（需 GPU 執行階段，D-024）。第一格的 `BASE` 選預訓練模型（D-021：`macbert`、`ckip`），兩個各執行一次；每次產生 fp32 與 int8 兩個模型包。

步驟：1. 設定與檢查 2. 上傳訓練資料（只在工作階段，不掛 Drive） 3. fine-tune（每個 epoch 存 checkpoint，可續跑） 4. 匯出 ONNX 並驗證 5. int8 量化並驗證 6. 打包 7. 印出 SHA-256。

注意：
- 第一格若安裝了不同版本的套件，請「重新啟動執行階段」後從第一格重跑。
- 工作階段中斷後檔案全部消失；模型包產生後立刻下載。
- notebook 進 git 時不帶輸出。

In [ ]:
# 套件版本（D-022；只在 Colab，不進 uv.lock）。裝了不同版本請重新啟動執行階段後重跑
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "torch==2.7.1",
        "torchvision==0.22.1",
        "torchaudio==2.7.1",
        "transformers==4.53.3",
        "optimum[onnxruntime]==1.27.0",
        "scikit-learn",
        "tokenizers",
    ],
    check=True,
)

import re  # noqa: E402

import torch  # noqa: E402
from packaging.version import Version  # noqa: E402

# 參數：BASE 選預訓練模型（macbert｜ckip）；revision 釘 commit hash（開發規格 7.27、D-025 B）
BASE = "macbert"
MAX_LEN = 512
SEED = 42

EPOCHS = 3
LEARNING_RATE = 2e-5
BATCH_SIZE = 8
GRAD_ACCUM = 2
VAL_FRACTION = 0.1
# 自行決定：int8 的驗證集 macro-F1 比 fp32 低超過這個值就印出警告（兩個包照常產生）
INT8_MAX_F1_DROP = 0.03

MACBERT_REV = "a986e004d2a7f2a1c2f5a3edef4e20604a974ed1"
# ckiplab/bert-base-chinese 的 refs/pr/1（safetensors 轉換 PR）commit；main 只有 pickle 權重
CKIP_PR_COMMIT = "d3e55b839fccbf1bba9e194b8626b18b30fcc5f3"
GOOGLE_TOKENIZER_REV = "8f23c25b06e129b6c986331a13d8d025a92cf0ea"
BASE_MODELS = {
    "macbert": {
        "model": "hfl/chinese-macbert-base",
        "model_rev": MACBERT_REV,
        "tokenizer": "hfl/chinese-macbert-base",
        "tokenizer_rev": MACBERT_REV,
    },
    "ckip": {
        "model": "ckiplab/bert-base-chinese",
        "model_rev": CKIP_PR_COMMIT,
        "tokenizer": "google-bert/bert-base-chinese",
        "tokenizer_rev": GOOGLE_TOKENIZER_REV,
    },
}

if BASE not in BASE_MODELS:
    raise SystemExit(f"BASE 必須是 {sorted(BASE_MODELS)}，目前是 {BASE!r}")
# macbert 只有 .bin（pickle）；torch>=2.6 的 torch.load 預設 weights_only=True 才能擋住惡意 pickle
if Version(torch.__version__.split("+")[0]) < Version("2.6"):
    raise SystemExit(f"需要 torch>=2.6，目前 {torch.__version__}；請重新啟動執行階段")

CFG = BASE_MODELS[BASE]
MODEL_ID, MODEL_REV = CFG["model"], CFG["model_rev"]
TOK_ID, TOK_REV = CFG["tokenizer"], CFG["tokenizer_rev"]
for rev in (MODEL_REV, TOK_REV):
    # revision 還是佔位字串（<revision>）或不是 40 位 commit hash 就停止
    if rev.startswith("<") or not re.fullmatch(r"[0-9a-f]{40}", rev):
        raise SystemExit(f"revision 仍是佔位字串或格式不對：{rev!r}")
print(f"BASE={BASE} model={MODEL_ID}@{MODEL_REV} tokenizer={TOK_ID}@{TOK_REV} max_len={MAX_LEN}")

## 2. 上傳訓練資料（D-024：只在這個工作階段，不掛 Drive）

選取 `data/datasets/<名稱>.jsonl` 與同名的 `.meta.json` 兩個檔。

In [ ]:
import json
import random
from collections import Counter

import numpy as np
from google.colab import files
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

# 順序必須與 radar.common.enums.Polarity 相同（ONNX logits 的類別順序，單元測試比對）
LABELS = ["positive", "negative", "neutral"]
LABEL2ID = {name: i for i, name in enumerate(LABELS)}
ID2LABEL = dict(enumerate(LABELS))

uploaded = files.upload()
meta_names = [n for n in uploaded if n.endswith(".meta.json")]
data_names = [n for n in uploaded if n.endswith(".jsonl")]
if len(meta_names) != 1 or len(data_names) != 1:
    raise ValueError("請剛好上傳一個 .jsonl 與一個 .meta.json")
meta = json.loads(uploaded[meta_names[0]])
for key in ("dataset", "labeler", "label_version", "max_chars"):
    if key not in meta:
        raise ValueError(f"meta 缺欄位 {key}")

rows = [json.loads(line) for line in uploaded[data_names[0]].decode("utf-8").splitlines() if line]
texts = [r["text"] for r in rows]
y = [LABEL2ID[r["polarity"]] for r in rows]  # 未知類別會 KeyError
if any(not isinstance(t, str) or not t.strip() for t in texts):
    raise ValueError("有空白或非字串的 text")
counts = Counter(r["polarity"] for r in rows)
print(f"{len(rows)} 筆；各類別 {dict(counts)}")
if min(counts.get(name, 0) for name in LABELS) < 10:
    raise ValueError("每個類別至少要 10 筆")

# 驗證集：分層 10%，固定 seed
train_idx, val_idx = train_test_split(
    list(range(len(rows))), test_size=VAL_FRACTION, random_state=SEED, stratify=y
)
train_texts = [texts[i] for i in train_idx]
train_y = [y[i] for i in train_idx]
val_texts = [texts[i] for i in val_idx]
val_y = [y[i] for i in val_idx]
train_counts = Counter(ID2LABEL[i] for i in train_y)
print(f"train {len(train_idx)}、val {len(val_idx)}")

## 3. fine-tune

載入釘 revision 的 tokenizer 與權重（`trust_remote_code=False`）：`ckip` 只載 safetensors，`macbert` 載 `.bin`（D-025 B）。

In [ ]:
import os
import time

from transformers import (
    AutoModelForSequenceClassification,
    BertTokenizerFast,
    DataCollatorWithPadding,
    Trainer,
    TrainerCallback,
    TrainingArguments,
    set_seed,
)
from transformers.trainer_utils import get_last_checkpoint

if not torch.cuda.is_available():
    raise SystemExit("沒有 GPU：請選「執行階段 > 變更執行階段類型」的 GPU")
set_seed(SEED)

tokenizer = BertTokenizerFast.from_pretrained(TOK_ID, revision=TOK_REV)


def load_model():
    kwargs = {
        "revision": MODEL_REV,
        "num_labels": len(LABELS),
        "id2label": ID2LABEL,
        "label2id": LABEL2ID,
    }
    if BASE == "ckip":
        return AutoModelForSequenceClassification.from_pretrained(
            MODEL_ID, use_safetensors=True, trust_remote_code=False, **kwargs
        )
    return AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID, use_safetensors=False, trust_remote_code=False, **kwargs
    )


model = load_model()


class EncodedDataset(torch.utils.data.Dataset):
    def __init__(self, text_list, label_list):
        self.enc = tokenizer(text_list, truncation=True, max_length=MAX_LEN)
        self.labels = label_list

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = self.labels[i]
        return item


train_ds = EncodedDataset(train_texts, train_y)
val_ds = EncodedDataset(val_texts, val_y)

# 與 baseline 的 class_weight=balanced 同義
CLASS_WEIGHTS = torch.tensor(
    [len(train_y) / (len(LABELS) * max(1, train_counts.get(name, 0))) for name in LABELS],
    dtype=torch.float32,
)

TIME_CHECK_STEP = 50


def estimate_total_seconds(elapsed, steps_done, total_steps):
    return elapsed / steps_done * total_steps


def check_time_budget(total_seconds, limit=10 * 3600):
    if total_seconds > limit:
        raise RuntimeError(
            f"預估 {total_seconds / 3600:.1f} 小時，超過 10 小時（Colab 上限 12 小時）；"
            "請把第一格的 MAX_LEN 改成 256 後從第一格重跑"
        )


class TimeBudgetCallback(TrainerCallback):
    def on_train_begin(self, args, state, control, **kwargs):
        self.start = time.time()
        self.first_step = state.global_step

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step - self.first_step == TIME_CHECK_STEP:
            elapsed = time.time() - self.start
            remaining_steps = state.max_steps - self.first_step
            total = estimate_total_seconds(elapsed, TIME_CHECK_STEP, remaining_steps)
            print(f"前 {TIME_CHECK_STEP} 步 {elapsed:.0f} 秒；預估總時間 {total / 3600:.1f} 小時")
            check_time_budget(total)


class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        model_inputs = {k: v for k, v in inputs.items() if k != "labels"}
        outputs = model(**model_inputs)
        logits = outputs.logits.float()
        weight = CLASS_WEIGHTS.to(logits.device)
        loss = torch.nn.functional.cross_entropy(logits, inputs["labels"], weight=weight)
        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return {"macro_f1": f1_score(labels, preds, average="macro")}


# 目錄帶 BASE 與 MAX_LEN，換模型或長度重跑時不會讀到別次的 checkpoint
RUN_NAME = f"{BASE}-len{MAX_LEN}"
CKPT_DIR = f"/content/checkpoints/{RUN_NAME}"
FINAL_DIR = f"/content/finetuned/{RUN_NAME}"
args = TrainingArguments(
    output_dir=CKPT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.06,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=True,
    seed=SEED,
    report_to="none",
)
trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[TimeBudgetCallback()],
)
# 同一個工作階段內中斷後重跑這格，會從最後一個 epoch 的 checkpoint 續跑
last_checkpoint = get_last_checkpoint(CKPT_DIR) if os.path.isdir(CKPT_DIR) else None
print("resume from checkpoint:", last_checkpoint)
trainer.train(resume_from_checkpoint=last_checkpoint)
val_metrics = trainer.evaluate()
VAL_MACRO_F1 = float(val_metrics["eval_macro_f1"])
print("PyTorch 驗證集 macro-F1:", VAL_MACRO_F1)
trainer.save_model(FINAL_DIR)

## 4. 匯出 ONNX 並驗證

依開發規格 7.27：tokenizer 以 `BertTokenizerFast` 存出 `tokenizer.json`，用推論端的載入方式重新編碼，ids 必須與訓練用的完全一致；PyTorch 與 ONNX 的 logits 最大絕對差必須小於 1e-3，通過才打包。

In [ ]:
import subprocess
from pathlib import Path

import onnxruntime as ort
from tokenizers import Tokenizer

WORK = Path(f"/content/work/{RUN_NAME}")
ONNX_DIR = WORK / "onnx_fp32"
TOK_DIR = WORK / "tokenizer"
for d in (ONNX_DIR, TOK_DIR):
    d.mkdir(parents=True, exist_ok=True)

subprocess.run(
    [
        "optimum-cli",
        "export",
        "onnx",
        "--model",
        FINAL_DIR,
        "--task",
        "text-classification",
        str(ONNX_DIR),
    ],
    check=True,
)
tokenizer.save_pretrained(str(TOK_DIR))
TOKENIZER_JSON = TOK_DIR / "tokenizer.json"
FP32_ONNX = ONNX_DIR / "model.onnx"
if not TOKENIZER_JSON.exists() or not FP32_ONNX.exists():
    raise FileNotFoundError("缺少 tokenizer.json 或 model.onnx")

# 推論端的載入方式
tk = Tokenizer.from_file(str(TOKENIZER_JSON))
tk.enable_truncation(max_length=MAX_LEN)
PAD_ID = tk.token_to_id("[PAD]")
if PAD_ID is None:
    raise ValueError("tokenizer.json 沒有 [PAD]")


def encode_ids(text_list):
    return [enc.ids for enc in tk.encode_batch(text_list)]


rng = random.Random(SEED)
sample_idx = rng.sample(range(len(val_texts)), min(64, len(val_texts)))
sample_texts = [val_texts[i] for i in sample_idx]
hf_ids = tokenizer(sample_texts, truncation=True, max_length=MAX_LEN)["input_ids"]
tk_ids = encode_ids(sample_texts)
if hf_ids != tk_ids:
    raise AssertionError(
        "tokenizers.Tokenizer.from_file 編碼的 input_ids 與訓練用 tokenizer 不一致"
    )
print("tokenizer.json 重新編碼的 ids 與訓練用一致")


def pad_batch(id_lists, idx, pad_id):
    width = max(len(id_lists[i]) for i in idx)
    ids = np.full((len(idx), width), pad_id, dtype=np.int64)
    mask = np.zeros((len(idx), width), dtype=np.int64)
    for row, i in enumerate(idx):
        ids[row, : len(id_lists[i])] = id_lists[i]
        mask[row, : len(id_lists[i])] = 1
    return ids, mask


def run_onnx(session, id_lists, batch_size=16):
    input_names = {i.name for i in session.get_inputs()}
    order = sorted(range(len(id_lists)), key=lambda i: len(id_lists[i]))
    out = np.zeros((len(id_lists), len(LABELS)), dtype=np.float32)
    for start in range(0, len(order), batch_size):
        idx = order[start : start + batch_size]
        ids, mask = pad_batch(id_lists, idx, PAD_ID)
        feed = {"input_ids": ids, "attention_mask": mask}
        if "token_type_ids" in input_names:
            feed["token_type_ids"] = np.zeros_like(ids)
        out[idx] = session.run(["logits"], feed)[0]
    return out


def run_torch(id_lists, batch_size=16):
    model.eval()
    device = next(model.parameters()).device
    out = np.zeros((len(id_lists), len(LABELS)), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, len(id_lists), batch_size):
            idx = list(range(start, min(start + batch_size, len(id_lists))))
            ids, mask = pad_batch(id_lists, idx, PAD_ID)
            logits = model(
                input_ids=torch.from_numpy(ids).to(device),
                attention_mask=torch.from_numpy(mask).to(device),
            ).logits
            out[idx] = logits.float().cpu().numpy()
    return out


LOGITS_TOLERANCE = 1e-3
fp32_session = ort.InferenceSession(str(FP32_ONNX), providers=["CPUExecutionProvider"])
torch_logits = run_torch(tk_ids)
onnx_logits = run_onnx(fp32_session, tk_ids)
max_diff = float(np.abs(torch_logits - onnx_logits).max())
print(f"PyTorch 與 ONNX logits 最大絕對差: {max_diff:.2e}（門檻 {LOGITS_TOLERANCE}）")
if max_diff >= LOGITS_TOLERANCE:
    raise AssertionError("fp32 ONNX 與 PyTorch 的 logits 差異過大，不打包")

val_ids = encode_ids(val_texts)
fp32_pred = run_onnx(fp32_session, val_ids).argmax(axis=1)
FP32_ONNX_F1 = float(f1_score(val_y, fp32_pred, average="macro"))
print(f"fp32 ONNX 驗證集 macro-F1: {FP32_ONNX_F1:.4f}（PyTorch {VAL_MACRO_F1:.4f}）")

## 5. int8 動態量化並驗證

兩個包都會產生（7.27）；驗證集 macro-F1 比 fp32 低超過 `INT8_MAX_F1_DROP` 時印出警告，並把差值記進 int8 的 card，由甲方評估時判斷。

In [ ]:
from optimum.onnxruntime import ORTQuantizer
from optimum.onnxruntime.configuration import AutoQuantizationConfig

INT8_DIR = WORK / "onnx_int8"
INT8_DIR.mkdir(parents=True, exist_ok=True)
# 推論主機是 arm64（開發規格 7.27 測試段），動態量化不需要校正資料
qconfig = AutoQuantizationConfig.arm64(is_static=False, per_channel=False)
quantizer = ORTQuantizer.from_pretrained(str(ONNX_DIR), file_name="model.onnx")
quantizer.quantize(save_dir=str(INT8_DIR), quantization_config=qconfig)
INT8_ONNX = INT8_DIR / "model_quantized.onnx"
if not INT8_ONNX.exists():
    raise FileNotFoundError("量化沒有產出 model_quantized.onnx")

int8_session = ort.InferenceSession(str(INT8_ONNX), providers=["CPUExecutionProvider"])
int8_pred = run_onnx(int8_session, val_ids).argmax(axis=1)
INT8_F1 = float(f1_score(val_y, int8_pred, average="macro"))
drop = FP32_ONNX_F1 - INT8_F1
print(f"int8 驗證集 macro-F1: {INT8_F1:.4f}（fp32 {FP32_ONNX_F1:.4f}，差 {drop:.4f}）")
if drop > INT8_MAX_F1_DROP:
    print(f"警告：int8 比 fp32 低超過 {INT8_MAX_F1_DROP}，評估時請特別留意")

## 6. 打包

每個模型包只有 `model.onnx`、`tokenizer.json`、`model_card.json` 三個檔，`model_card.files` 記前兩者的 SHA-256。

In [ ]:
import hashlib
import shutil
import zipfile
from datetime import UTC, datetime

now = datetime.now(UTC)  # UTC
MODEL_VERSION = f"bert-{BASE}-{now:%Y%m%d%H%M}"
OUT_DIR = Path("/content/packages")
OUT_DIR.mkdir(parents=True, exist_ok=True)


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def build_package(model_version, onnx_src, quantization, val_f1, f1_drop=0.0):
    pkg_dir = WORK / model_version
    pkg_dir.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(onnx_src, pkg_dir / "model.onnx")
    shutil.copyfile(TOKENIZER_JSON, pkg_dir / "tokenizer.json")
    card = {
        "model_version": model_version,
        "created_at": now.isoformat(),
        "dataset": meta["dataset"],
        "labeler": meta["labeler"],
        "label_version": meta["label_version"],
        "max_chars": meta["max_chars"],
        "n_train": len(train_y),
        "class_counts": dict(train_counts),
        "params": {
            "tokenizer": f"{TOK_ID}@{TOK_REV}",
            "epochs": EPOCHS,
            "learning_rate": LEARNING_RATE,
            "batch_size": BATCH_SIZE * GRAD_ACCUM,
            "seed": SEED,
            "class_weight": "balanced",
            "quantization": quantization,
            "val_macro_f1": round(val_f1, 4),
            "val_f1_drop_vs_fp32": round(f1_drop, 4),
            "torch": torch.__version__,
        },
        "kind": "onnx",
        "base_model": f"{MODEL_ID}@{MODEL_REV}",
        "max_len": MAX_LEN,
        "labels": LABELS,
        "files": {
            "model.onnx": sha256_file(pkg_dir / "model.onnx"),
            "tokenizer.json": sha256_file(pkg_dir / "tokenizer.json"),
        },
    }
    (pkg_dir / "model_card.json").write_text(
        json.dumps(card, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    zip_path = OUT_DIR / f"{model_version}.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for name in ("model.onnx", "tokenizer.json", "model_card.json"):
            zf.write(pkg_dir / name, arcname=name)
    return zip_path


packages = [
    build_package(MODEL_VERSION, FP32_ONNX, "none", FP32_ONNX_F1),
    build_package(f"{MODEL_VERSION}-int8", INT8_ONNX, "dynamic_int8_arm64", INT8_F1, drop),
]
print("產生:", [p.name for p in packages])

## 7. 印出 SHA-256 並下載

把這裡印出的值與匯入時（`import_model`）核對；工作階段結束檔案就消失，請立刻下載。

下載失敗時，從左側檔案面板下載 `/content/packages/` 內的 zip。

In [ ]:
for zip_path in packages:
    print(f"{zip_path.name}  size={zip_path.stat().st_size}  sha256={sha256_file(zip_path)}")
for zip_path in packages:
    files.download(str(zip_path))